**TESTING 1 FILE DATA EXTRACTION**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np

def inspect_npy_and_txt(npy_path, txt_path):
    print(f"Reading: {npy_path}")
    try:
        npy_data = np.load(npy_path)
        print("Shape of NPY data:", npy_data.shape)
        print("Sample of NPY data:")
        print(npy_data if npy_data.size < 50 else npy_data.flatten()[:50])
    except Exception as e:
        print("Error reading .npy file:", e)

    print("\nReading:", txt_path)
    try:
        with open(txt_path, 'r') as f:
            txt_data = f.read()
        print("Content of TXT file:")
        print(txt_data)
    except Exception as e:
        print("Error reading .txt file:", e)

# Replace these with actual paths before running
npy_path = '/content/drive/MyDrive/CT-MRI Paired Dataset/Saved Models/weights_dump/layer_1_weight_0.npy'
txt_path = '/content/drive/MyDrive/CT-MRI Paired Dataset/Saved Models/weights_dump/layer_1_weight_0.txt'

inspect_npy_and_txt(npy_path, txt_path)


**PYTHON SCRIPT TO CONVERT ALL .npy FILES INTO 'C'**

In [ ]:
import os
import numpy as np

# Hardcoded paths for your use case
npy_dir = "/content/drive/MyDrive/CT-MRI Paired Dataset/Saved Models/weights_dump"
header_dir = "/content/drive/MyDrive/CT-MRI Paired Dataset/Saved Models/headers"

def npy_to_header(npy_path, header_dir):
    arr = np.load(npy_path)
    flat = arr.flatten()
    shape = arr.shape

    basename = os.path.splitext(os.path.basename(npy_path))[0]
    var_name = basename
    header_path = os.path.join(header_dir, basename + ".h")

    os.makedirs(header_dir, exist_ok=True)

    guard = f"_{var_name.upper()}_H_"
    with open(header_path, 'w') as f:
        f.write(f"#ifndef {guard}\n#define {guard}\n\n")

        for idx, dim in enumerate(shape):
            f.write(f"#define {var_name.upper()}_DIM{idx} {dim}\n")
        f.write("\n")

        total = flat.size
        f.write(f"static const float {var_name}_data[{total}] = {{\n")
        for i, val in enumerate(flat):
            f.write(f"{val:.6f}f")
            if i < total - 1:
                f.write(", ")
            if (i + 1) % 10 == 0:
                f.write("\n")
        f.write("\n};\n\n")

        f.write(f"#endif // {guard}\n")

    print(f"Generated header: {header_path}")

def convert_npy_directory(npy_dir, header_dir):
    for root, _, files in os.walk(npy_dir):
        for filename in files:
            if filename.endswith(".npy"):
                npy_path = os.path.join(root, filename)
                npy_to_header(npy_path, header_dir)

convert_npy_directory(npy_dir, header_dir)
print("✅ All .npy files have been converted to C headers.")


**VERIFYING THE GENERATED .h FILES IF THEY ARE OKAY OR NOT**

In [ ]:
import numpy as np
import os
import re

npy_dir = "/content/drive/MyDrive/CT-MRI Paired Dataset/Saved Models/weights_dump"
header_dir = "/content/drive/MyDrive/CT-MRI Paired Dataset/Saved Models/headers"

def verify_header_against_npy(npy_path, header_path):
    original_array = np.load(npy_path)
    original_flat = original_array.flatten()

    with open(header_path, 'r') as f:
        content = f.read()

    dims = [int(val) for val in re.findall(r"#define\s+\w+_DIM\d+\s+(\d+)", content)]
    expected_shape = tuple(dims)

    data_match = re.search(r"=\s*{([^}]+)};", content, re.DOTALL)
    if not data_match:
        print(f"ERROR: No data array found in {header_path}")
        return False

    float_values = [float(v.replace('f', '').strip()) for v in data_match.group(1).split(',') if v.strip()]

    if len(float_values) != original_flat.size:
        print(f"ERROR: Mismatch in data length in {header_path}")
        return False

    npy_vals = np.round(original_flat, 6)
    header_vals = np.round(np.array(float_values), 6)
    if not np.allclose(npy_vals, header_vals, atol=1e-5):
        print(f"ERROR: Values don't match in {header_path}")
        return False

    if expected_shape != original_array.shape:
        print(f"ERROR: Shape mismatch in {header_path} (expected {original_array.shape}, got {expected_shape})")
        return False

    print(f"✅ Verified: {os.path.basename(header_path)}")
    return True

# Loop through all .npy and .h file pairs
for filename in os.listdir(npy_dir):
    if filename.endswith('.npy'):
        npy_path = os.path.join(npy_dir, filename)
        header_path = os.path.join(header_dir, filename.replace('.npy', '.h'))
        verify_header_against_npy(npy_path, header_path)


**CONVERSION OF .txt FILES INTO 'C' FILES**

In [ ]:
import os

txt_folder = "/content/drive/MyDrive/CT-MRI Paired Dataset/Saved Models/weights_dump"
h_folder = "/content/drive/MyDrive/CT-MRI Paired Dataset/Saved Models/headers_txt"
os.makedirs(h_folder, exist_ok=True)

for filename in os.listdir(txt_folder):
    if filename.endswith(".txt"):
        txt_path = os.path.join(txt_folder, filename)
        h_filename = filename.replace(".txt", ".h")
        h_path = os.path.join(h_folder, h_filename)

        with open(txt_path, "r") as f:
            try:
                weights = [float(line.strip()) for line in f if line.strip()]
            except ValueError:
                print(f"[SKIP] Invalid float in: {filename}")
                continue

        var_name = h_filename.replace(".h", "")
        with open(h_path, "w") as f:
            f.write(f"float {var_name}[] = {{\n")
            for i, w in enumerate(weights):
                end = ",\n" if i < len(weights) - 1 else "\n"
                f.write(f"    {w}{end}")
            f.write("};\n")

        # Debug print (optional)
        print(f"[OK] {h_filename} - Written {len(weights)} values")


**VERIFICATION OF THE GENERATED .h FILES FROM THE .txt FILES**

In [ ]:
import os
import re

txt_folder = "/content/drive/MyDrive/CT-MRI Paired Dataset/Saved Models/weights_dump"
h_folder = "/content/drive/MyDrive/CT-MRI Paired Dataset/Saved Models/headers_txt"

missing_h_files = []
length_mismatches = []
invalid_h_files = []

def count_floats_from_txt(path):
    with open(path, "r") as f:
        lines = [line.strip() for line in f if line.strip()]
    return len(lines)

def count_floats_from_h(path):
    with open(path, "r") as f:
        content = f.read()

    # Extract float values from C array: float name[] = { ... };
    array_match = re.search(r"\{([^}]+)\}", content, re.DOTALL)
    if not array_match:
        return -1  # Invalid structure

    array_str = array_match.group(1)
    values = array_str.split(',')
    floats = []
    for val in values:
        val = val.strip()
        if val:
            try:
                floats.append(float(val))
            except ValueError:
                return -2  # Malformed number
    return len(floats)

# Iterate over all .txt files
for filename in sorted(os.listdir(txt_folder)):
    if not filename.endswith(".txt"):
        continue

    base = filename.replace(".txt", "")
    txt_path = os.path.join(txt_folder, filename)
    h_path = os.path.join(h_folder, base + ".h")

    if not os.path.exists(h_path):
        missing_h_files.append(filename)
        continue

    txt_count = count_floats_from_txt(txt_path)
    h_count = count_floats_from_h(h_path)

    if h_count == -1:
        invalid_h_files.append((filename, "Invalid C array format"))
    elif h_count == -2:
        invalid_h_files.append((filename, "Non-numeric value in .h"))
    elif txt_count != h_count:
        length_mismatches.append((filename, txt_count, h_count))

# Report
print("=== VERIFICATION REPORT ===")
if not missing_h_files and not length_mismatches and not invalid_h_files:
    print("✅ All .h files match their .txt sources exactly.")
else:
    if missing_h_files:
        print("\n[MISSING .h FILES]")
        for file in missing_h_files:
            print(f" - {file}")

    if length_mismatches:
        print("\n[LENGTH MISMATCHES]")
        for file, txt_len, h_len in length_mismatches:
            print(f" - {file}: txt({txt_len}) vs h({h_len})")

    if invalid_h_files:
        print("\n[INVALID .h FILES]")
        for file, reason in invalid_h_files:
            print(f" - {file}: {reason}")


**Model Architecture Extraction From .h5 File**

In [ ]:
import h5py
import tensorflow as tf
import json
from google.colab import drive
import os

# Mount Google Drive (skip if already mounted)
drive_path = "/content/drive"
if not os.path.exists(drive_path):
    drive.mount(drive_path)
else:
    print("Drive already mounted at /content/drive")

# Path to the .h5 file
h5_path = "/content/drive/MyDrive/CT-MRI Paired Dataset/Saved Models/trainedpix2pix.h5"

# Method 1: Extract JSON config using Keras
try:
    # Define custom resize function
    @tf.keras.utils.register_keras_serializable()
    def resize_function(x):
        return tf.image.resize(x, [256, 256], method='bilinear')

    # Load model
    model = tf.keras.models.load_model(
        h5_path,
        custom_objects={'Custom>resize_function': resize_function}
    )

    # Get JSON config
    model_config = model.get_config()
    with open('/content/drive/MyDrive/CT-MRI Paired Dataset/Saved Models/model_config.json', 'w') as f:
        json.dump(model_config, f, indent=2)
    print("Saved model config to model_config.json")

    # Print summary for quick check
    model.summary()
except Exception as e:
    print(f"Error loading model: {e}")

# Method 2: Extract JSON config using h5py
try:
    with h5py.File(h5_path, 'r') as f:
        model_config = f.attrs.get('model_config')
        if model_config:
            model_config = json.loads(model_config)
            with open('/content/drive/MyDrive/CT-MRI Paired Dataset/Saved Models/h5py_model_config.json', 'w') as f:
                json.dump(model_config, f, indent=2)
            print("Saved h5py model config to h5py_model_config.json")
        else:
            print("No model_config found in .h5 file")
except Exception as e:
    print(f"Error reading .h5 file: {e}")

In [ ]:
import nibabel as nib
import numpy as np

# === Step 1: Load the .nii file ===
nii_path = '/content/drive/MyDrive/CT-MRI Paired Dataset/CT/sub-0001_ct.nii.gz'  # Replace with the path to your .nii file
img = nib.load(nii_path)
data = img.get_fdata()

# === Step 2: Print details ===
print(f"Data shape: {data.shape}")
print(f"Data type: {data.dtype}")
print(f"Min pixel value: {np.min(data)}")
print(f"Max pixel value: {np.max(data)}")
print(f"Mean pixel value: {np.mean(data)}")

# === Step 3: Visualize central slice (optional) ===
import matplotlib.pyplot as plt

middle_slice = data.shape[2] // 2
plt.imshow(data[:, :, middle_slice], cmap='gray')
plt.title("Middle Slice")
plt.colorbar()
plt.show()


In [ ]:
import numpy as np
import nibabel as nib

def convert_nii_to_h(nii_path, output_h_path, slice_index=None):
    # Load the .nii image
    img = nib.load(nii_path)
    data = img.get_fdata()

    # Pick the middle slice or a specific one
    if slice_index is None:
        slice_index = data.shape[2] // 2
    slice_data = data[:, :, slice_index]

    # Normalize to [0, 1]
    slice_data = (slice_data + 1024.0) / (3071.0 + 1024.0)

    # Flatten and convert to C array
    flat_data = slice_data.flatten()
    with open(output_h_path, 'w') as f:
        f.write("#ifndef CT_IMAGE_SLICE_H\n#define CT_IMAGE_SLICE_H\n\n")
        f.write("float ct_image_slice[{}] = {{\n".format(flat_data.size))
        for i, val in enumerate(flat_data):
            f.write(f"{val:.6f}")
            if i != flat_data.size - 1:
                f.write(", ")
            if (i + 1) % 8 == 0:
                f.write("\n")
        f.write("\n};\n\n#endif")
    print(f"[✓] CT slice saved to {output_h_path}")

# Example usage
convert_nii_to_h(
    nii_path="/content/drive/MyDrive/CT-MRI Paired Dataset/CT/sub-0001_ct.nii.gz",
    output_h_path="/content/drive/MyDrive/CT-MRI Paired Dataset/Saved Models/h files/input CT .h files/sub-0001_ct.h"
)


In [ ]:
import os

# Path to the pipeline folder
pipeline_folder = '/content/drive/MyDrive/CT-MRI Paired Dataset/Saved Models/h files/pipeline'

# Iterate over all files in the folder
for filename in os.listdir(pipeline_folder):
    if filename.endswith('.h') and 'weight' in filename:
        new_filename = filename.replace('weight', 'pipeline')
        src = os.path.join(pipeline_folder, filename)
        dst = os.path.join(pipeline_folder, new_filename)
        os.rename(src, dst)
        print(f'Renamed: {filename} → {new_filename}')
